In [ ]:
import czifile
import os
from lxml import etree
import seaborn as sns
import pandas as pd

In [ ]:
file = "/Volumes/Intenso/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_images/CRISPR-Exp5_IHC-Exp2_Brain-5_section-1_488-VGLUT1_647-PSD95_VCAM1-gRNA_63X&3XzoomAiryscan_CA3_SR.czi"

czi = czifile.CziFile(file)
czi_xml_str = czi.metadata() # gets the metadata in a xml string format
czi_parsed = etree.fromstring(czi_xml_str) # parses the czi_xml_str file

In [ ]:
print(file)
print(czi_xml_str)

In [ ]:
superres_params = czi_parsed.findall(".//SuperResolutionParameter")

ch_1_superresparam_value = float(superres_params[0].text)
ch_2_superresparam_value = float(superres_params[1].text)


print(ch_1_superresparam_value, ch_2_superresparam_value )

In [ ]:
# making a loop to get the supperres_params for each image for VCAM1, VGLUT1-PSD95 condition

# input folder
input_folder = "/Volumes/Intenso/image-analysis/synapse-counting/IHC_Exp9_VCAM1_VGLUT1-PSD95/airyscan_images/"

# get a list of files in that input_folder
file_list = os.listdir(input_folder)

print(file_list)

superres_params_list = []

for file in file_list:
    long_file = input_folder + file
    czi = czifile.CziFile(long_file)
    czi_xml_str = czi.metadata()
    czi_parsed = etree.fromstring(czi_xml_str)
    superres_params = czi_parsed.findall(".//SuperResolutionParameter")

    ch_1_superresparam_value = float(superres_params[0].text)
    ch_2_superresparam_value = float(superres_params[1].text)

    superres_params_list.append({"img_filename": file, "ch_1_superresparam_value": ch_1_superresparam_value, "ch_2_superresparam_value": ch_2_superresparam_value})

print(superres_params_list)


superres_params_df = pd.DataFrame(superres_params_list)
superres_params_df

In [ ]:
superres_params_df["section"] =superres_params_df['img_filename'].str.replace(r'_[^_]+-gRNA_', '_', regex=True)
superres_params_df["gRNA"] = superres_params_df['img_filename'].apply(lambda x: x.split('_')[-4:-3]).apply(lambda x: '_'.join(x))


superres_params_df

In [ ]:
df = superres_params_df

In [ ]:
df_ordered = df.pivot_table(index='section', columns = "gRNA", values = ["ch_1_superresparam_value", "ch_2_superresparam_value"])
df_ordered.head(10)

In [ ]:
df_ordered["diff_ch1"] = df_ordered[('ch_1_superresparam_value',  'LacZ-gRNA')] - df_ordered[('ch_1_superresparam_value', 'VCAM1-gRNA')]
df_ordered["diff_ch2"] = df_ordered[('ch_2_superresparam_value',  'LacZ-gRNA')] - df_ordered[('ch_2_superresparam_value', 'VCAM1-gRNA')]
df_ordered

In [ ]:
df_ordered = df_ordered.reset_index().rename(columns={'index': 'section'})
df_ordered['hippocampal_layer'] = df_ordered['section'].apply(lambda x: x.split('_')[-2:]).apply(lambda x: ' '.join(x))
df_ordered

In [ ]:
sns.catplot(
    data=df_ordered, 
    x="hippocampal_layer", 
    y="diff_ch1",
    height=5,
    aspect=2
).set(
    title="VGLUT1 (ch1) super-resolution-param-differences [LacZ-gRNA - VCAM1-gRNA]"
)

In [ ]:
sns.catplot(
    data=df_ordered, 
    x="hippocampal_layer", 
    y="diff_ch2",
    height=5,
    aspect=2
).set(
    title="PSD95 (ch2) super-resolution-param-differences [LacZ-gRNA - VCAM1-gRNA]"
)
